In [ ]:
import pandas as pd 
import numpy as np 
import datetime 
import warnings
from sklearn.preprocessing import OneHotEncoder

This program provides aggregated make or miss by target date POF customer OTD

TOAD:

select *

from

EID.POF_CUST_OTD

where 

TARGET_DATE > TO_DATE('01/01/2022', 'MM/DD/YYYY')
;

In [19]:
aggregated_chunks = []

for chunk in pd.read_csv("POF_CUST_OTD.csv", delimiter='|', chunksize=100000):
    
    # 1. Convert date and clean keys immediately
    chunk['target_date'] = pd.to_datetime(chunk['target_date'], errors='coerce')
    
    # 2. Create the unique key properly within the chunk
    # We use .astype(str) on the specific columns without overwriting the whole chunk
    chunk['key'] = (
        chunk['sales_order'].astype(str) + 
        '.' + 
        chunk['so_line'].astype(str)
    )
    # 3. Deduplicate based on the unique key
    extract = chunk.drop_duplicates(subset=['key'])
    
    # 4. One-hot encode the 'make_miss' column
    # This creates 'make_miss_make' and 'make_miss_miss' (check your actual column names)
    extract = pd.get_dummies(chunk, columns=['make_miss'], dtype=int)
    
    # 5. Group by Year AND Month
    # We reference extract['target_date'] since extract is our working DataFrame
    chunk_grouped = extract.groupby([
        extract['target_date'].dt.year.rename('Year'), 
        extract['target_date'].dt.month.rename('Month')
    ]).sum(numeric_only=True)
    
    aggregated_chunks.append(chunk_grouped)

# 6. Final Aggregation
# Group by both levels (Year and Month) to merge results from all chunks
final_df = pd.concat(aggregated_chunks).groupby(level=[0, 1]).sum()

In [ ]:
final_df.reset_index().to_excel("agg_make_miss_POF_OTD.xlsx")